# NeuroScan AI: Exploración de Datos, Manifiesto y Auditoría de Fuga
## Módulo 3: Databricks e IA Aplicada - Proyecto Final

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EduardCY/m3-proyecto-final-databricks-ia/blob/main/notebooks/01_exploracion_eda.ipynb)

Este notebook implementa la **Etapa 1** del pipeline:
1. Verificación de integridad y descarte de imágenes corruptas.
2. Generación del manifiesto estructurado con `pandas` y el paso equivalente en `PySpark`.
3. Auditoría empírica de **fuga de datos (data leakage)** entre particiones.
4. Creación del split estratificado (70% train, 15% val, 15% test) guardado en `data/manifest_split.csv`.


In [ ]:
import sys
from pathlib import Path

# Configurar path raíz del proyecto
PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import CLASSES, DATA_DIR, MANIFEST_PATH, SPLIT_MANIFEST_PATH, SPLIT_RATIOS, SEED
from src.dataset import build_manifest, split_manifest, build_manifest_spark

print(f"Ruta de datos: {DATA_DIR}")
print(f"Clases a clasificar: {CLASSES}")


### 1. Construcción del Manifiesto con Auditoría de Integridad


In [ ]:
# Construir el manifiesto auditando imágenes corruptas
manifest_df = build_manifest(data_dir=DATA_DIR, use_cache=False)
print(f"Total imágenes válidas: {len(manifest_df)}")
manifest_df.head()


### 2. Distribución de Clases


In [ ]:
class_counts = manifest_df["label_name"].value_counts()
print("Distribución de imágenes por patología:")
print(class_counts)

plt.figure(figsize=(8, 4.5))
sns.barplot(x=class_counts.index, y=class_counts.values, palette="Blues_r")
plt.title("Distribución de Resonancias por Clase (Brain Tumor MRI)")
plt.xlabel("Diagnóstico")
plt.ylabel("Número de Imágenes")
plt.grid(axis='y', alpha=0.3)
plt.show()


### 3. Equivalente Simbólico en PySpark (Fase 2 Databricks)


In [ ]:
try:
    spark_df = build_manifest_spark(data_dir=DATA_DIR)
    print("Esquema generado en PySpark (Simulación de Unity Catalog):")
    spark_df.printSchema()
    spark_df.groupBy("raw_folder").count().show()
except Exception as e:
    print(f"Aviso PySpark (requiere Java/Spark en entorno local): {e}")


### 4. Partición Estratificada y Verificación de Fuga de Datos


In [ ]:
# Generar split 70/15/15 estratificado
split_df = split_manifest(manifest_df, split_ratios=SPLIT_RATIOS, seed=SEED)

train_paths = set(split_df[split_df["split"] == "train"]["path"])
val_paths = set(split_df[split_df["split"] == "val"]["path"])
test_paths = set(split_df[split_df["split"] == "test"]["path"])

# Auditoría estricta de intersección
leakage_train_val = len(train_paths.intersection(val_paths))
leakage_train_test = len(train_paths.intersection(test_paths))
leakage_val_test = len(val_paths.intersection(test_paths))

print(f"Fuga Train - Val: {leakage_train_val} imágenes")
print(f"Fuga Train - Test: {leakage_train_test} imágenes")
print(f"Fuga Val - Test: {leakage_val_test} imágenes")

assert leakage_train_val == 0 and leakage_train_test == 0 and leakage_val_test == 0, "¡Fuga de datos detectada!"
print("\n✅ COMPROBACIÓN EXITOSA: Partición sin fuga de datos confirmada y guardada en data/manifest_split.csv")
